# Procesador de Velocidad WAV (Remuestreo)

> **Nota técnica:** Este procesador modifica la velocidad de reproducción **por remuestreo**.
> Acelerar descarta muestras alternadas; desacelerar duplica cada muestra.
> Esto cambia simultáneamente la duración **y el tono (pitch)** del audio.
> Para cambiar velocidad sin afectar el pitch se requiere un algoritmo de
> *Time-Stretching* (ej. Phase Vocoder), que no está implementado aquí.

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import tkinter as tk
from tkinter import filedialog, messagebox

def cargar_archivo():
    ruta_archivo = filedialog.askopenfilename(filetypes=[("Archivos WAV", "*.wav")])
    if ruta_archivo:
        entry_ruta.delete(0, tk.END)
        entry_ruta.insert(0, ruta_archivo)

def procesar_audio(opcion):
    ruta_archivo = entry_ruta.get()

    if not ruta_archivo.lower().endswith('.wav'):
        messagebox.showerror("Error", "Por favor, selecciona un archivo .wav.")
        return

    frecuencia_muestreo, datos = wav.read(ruta_archivo)
    datos = np.array(datos)

    if opcion == "desacelerar":
        # Remuestreo: duplicar cada muestra → mitad de velocidad, pitch baja una octava
        datos_desacelerados = np.repeat(datos, 2, axis=0)
        wav.write("audio_desacelerado.wav", frecuencia_muestreo, datos_desacelerados)
        messagebox.showinfo("Éxito", "Audio desacelerado a 0.5x guardado.")

    elif opcion == "acelerar":
        # Remuestreo: tomar una muestra de cada dos → doble de velocidad, pitch sube una octava
        datos_acelerados = datos[::2]
        wav.write("audio_acelerado.wav", frecuencia_muestreo, datos_acelerados)
        messagebox.showinfo("Éxito", "Audio acelerado a x2 guardado.")

root = tk.Tk()
root.title("Procesador de Velocidad WAV (Remuestreo)")

tk.Label(root, text="Selecciona un archivo .wav:").pack(pady=10)
entry_ruta = tk.Entry(root, width=50)
entry_ruta.pack(pady=5)
tk.Button(root, text="Cargar Archivo", command=cargar_archivo).pack(pady=5)

tk.Label(root, text="Selecciona una opción:").pack(pady=10)
tk.Button(root, text="Acelerar a x2",     command=lambda: procesar_audio("acelerar")).pack(pady=5)
tk.Button(root, text="Desacelerar a 0.5x", command=lambda: procesar_audio("desacelerar")).pack(pady=5)

root.mainloop()
